In [23]:
%load_ext autoreload
%autoreload 1


from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd

%aimport lib.geoutils
from lib.geoutils import great_circle_distance_km

DATA_DIR = Path("inputs")
STATIONS_PATH = DATA_DIR / "ic-stations.parquet"
ROUTES_PATH = DATA_DIR/"ic-routes.parquet"

C_STN = "stn_id"
C_ROUTE = "route_id"
C_FUA = "fua_id"
C_ROUTESET = "routes"


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Notes to self: 
- Centroid distance is treated as unreliable, especially for polycentric FUAs such as Ruhrgebiet (ID 517)
- 

# IO operations


In [24]:
ic_stations = (
    gpd.read_parquet(STATIONS_PATH).drop(
        columns=["stop_id"],
        errors="ignore")  # drop list of raw GTFS stop IDs (not scalar key)
    .reset_index())
df_fuas = gpd.read_parquet(DATA_DIR / "fuas.parquet").set_index(C_FUA)

ic_routes = pd.read_parquet(ROUTES_PATH)
_shape1 = ic_routes.shape[0]
ic_routes = ic_routes[ic_routes.is_rail]
print(f"N_routes: {ic_routes.shape[0]} / {_shape1} rail routes")
ic_routes.head(1)

N_routes: 26576 / 36649 rail routes


,route_id,stn_id,agency,is_rail,ivt,feed_trip,toc,domicile,seg_id,len_km,speed
0,41,"[52, 569, 66, 1602, 2917, 14, 570, 4102, 6722,...",800151 DB Regio AG Nordost,True,148,"[[67, 77963]]",DB,DE,"[21209, 19590, 21498, 21496, 17038, 21453, 214...",206.219014,83.602303


In [25]:
# # short point check
# _surface_area = df_fuas.geometry.to_crs("EPSG:3035").area
# _temp = df_fuas.join(pd.Series((np.sqrt(_surface_area / np.pi) / 1000).rename("radius")))#.sort_values("radius")
# _temp = _temp.join(pd.Series(_surface_area / (1000 * 1000)).rename("area_km2"))
# _temp.sort_values("area_km2").drop(["centre", "geometry"], axis=1)

# Main stations per FUA, from number of unique routes


## Station importance by unique service route
- Get stations' importance from number of routes serviced (fraction of N_routes within 1 FUA)

## Enrich stations dataset

### Station-centroid distances 
(not used in analysis due to pop-weighted centroid unreliability, but potentially useful for access/egress)

In [26]:
_dists = ic_stations[[C_FUA, "geometry"]].join(df_fuas["centre"], on=C_FUA)
_dists["gcd_centroid"] = great_circle_distance_km(
    _dists["geometry"], _dists["centre"])
# _dists.head(2)
ic_stations = ic_stations.join(_dists["gcd_centroid"])

### Link country information to station level

In [27]:
ic_stations = ic_stations.join(df_fuas[["name", "icc"]], on=C_FUA)

### Station importance using number of serviced routes
- importance rank order: number of serviced routes, with ties broken by proximity to FUA centroid

In [28]:
# invert route->station 1:m mapping to station->route 1:m
_stn_to_routes = (
    ic_routes.set_index(C_ROUTE)  # set route_id as index
    [C_STN].explode()  # create 1:1 route-station rows
    .reset_index()  # route_id to column
    .groupby(C_STN)  # groupby station
    [C_ROUTE].apply(set)  # convert routes to set
    .rename(C_ROUTESET)  # set as Series name
)

# Join to stations: set of routes serviced
# (Note that stations without rail services are dropped here)
ic_stations = (ic_stations.join(
    _stn_to_routes, on=C_STN).dropna(axis=0, subset=[C_ROUTESET]))
# count number of serviced routes
ic_stations["n_routes"] = ic_stations[C_ROUTESET].apply(len)
# sort to establish importance rank
ic_stations = ic_stations.sort_values(
    # sort also FUA for performance
    [C_FUA, "n_routes", "gcd_centroid"], ascending=(True, False, True))

In [29]:
# pd.DataFrame(ic_stations)

In [30]:
ic_stations.set_index(C_STN, drop=False).to_parquet(
    DATA_DIR/"ic-stations2.parquet")

## (deprecated) Gather station importance for LD services 
_(note: this station importance section is deprecated for a simpler, sufficient scheme)_

Per FUA and per station, descending from the most-important station: 
- gather the set of unique contributing routes per station
- gather all stations that have any unique contribution as `core` stations
- calculate core station `weighted distances`, which is the weighted average of distances from a station to all other core stations. Used for access time/speed calculation and shortest-path inferral
    $$D_{i}^{w} = \sum_{j \in \mathcal{S_{core}}}{D_{ij} R_{j}} / \sum_{j \in \mathcal{S_{core}}} R_j $$
    - where $i$ = own station, $S_{core}$ is set of all core stations (including self), $D_{ij}$ is the distance between core stations, $R_j$ the route coverage ratio (overlap included)
    - in matrix form: $$\mathbf{d^{w}} = \frac{1}{\mathbf{1}^T R}(\mathbf{DR})$$ 



In [31]:
# _collector = []
# for _fua, _sdf in ic_stations.groupby(C_FUA, sort=False):
#     # get superset of unique rail routes serviced by stations in this FUA
#     # _superset = set().union(*_sdf[C_ROUTESET])
#     # _superset_ref = _superset.copy()
#     # _len_superset_ref = len(_superset_ref)


#     # # Greedy Set Cover loop: calculate marginal route utility per station
#     # _coverage_set = set()
#     # _coverage_deltas = []
#     # _nroutes_covered = []
#     # for routeset in _sdf[C_ROUTESET]:
#     #     _coverage = _coverage_set | routeset
#     #     _routes_added = routeset - _coverage_set
#     #     # normalise the marginal added utility vs total FUA routeset
#     #     _coverage_deltas.append(len(_routes_added) / _len_superset_ref)
#     #     _nroutes_covered.append(len(_routes_added))
#     #     _coverage_set = _coverage
#     # _sdf["marginal_coverage_delta"] = _coverage_deltas  # key field
#     # _sdf["added_nroutes"] = _nroutes_covered # can be deprecated
#     # _sdf = _sdf.sort_values("coverage_delta", ascending=False)
#     # _sdf["coverage"] = _sdf["coverage_delta"].cumsum()

#     # # Calculate fraction of routes covered per station
#     # _sdf["frac_routes"] = _sdf["n_routes"] / _len_superset_ref
#     # # Set stations with any marginal utility as core
#     # _sdf["is_core"] = _sdf["marginal_coverage_delta"] > 0  # key field

#     # Calculate route-weighted distance between core stations and all
#     for _stations, _target in zip([_sdf, _sdf[_sdf.is_core].copy()], ["rwd_all","rwd_core"]):
#         # _core_stns = _sdf[_sdf.is_core].copy()
#         # get interstation GCD as square matrix
#         _dist_mat = great_circle_distance_km(_stations["geometry"],
#                                             _stations["geometry"],
#                                             comb=True)
#         _frac_routes = _stations.frac_routes.to_numpy()
#         _sdf[_target] = pd.Series(dtype="float64")  # for float column initialise
#         _sdf.loc[_stations.index, _target] = np.matmul(
#             _dist_mat, _frac_routes) / _frac_routes.sum() #/ _stations.shape[0]

#     # save full subdf
#     _collector.append(_sdf)

# ic_stations2 = pd.concat(_collector)

In [32]:
# ic_stations2.set_index(C_STN, drop=False).to_parquet(DATA_DIR/"ic-stations2.parquet")